# OptiPrime scoring with PEGG

**New in PEGG 3.1.**

This notebook scores pegRNAs with [OptiPrime](https://github.com/alvin-hsu/optiprime-src) (Hsu et al., *Nature Biotechnology* 2026) instead of PEGG's built-in `PEGG2_Score`.

**Why use OptiPrime?** `PEGG2_Score` is a hand-weighted combination of design features (GC content, PAM disruption, homology arm length). It is fast and needs no extra installation, but the weights were chosen by hand rather than fitted to data. OptiPrime is a machine learning model trained on large-scale measured prime editing outcomes, and it explicitly models **mismatch repair (MMR)** — often the dominant factor in whether an edit is installed. For a library you intend to order and screen, we recommend OptiPrime.

**The only cost is runtime:** roughly 14 s of fixed start-up plus ~0.1 s per pegRNA on one CPU core.

**Full documentation:** [pegg30.readthedocs.io/en/latest/optiprime.html](https://pegg30.readthedocs.io/en/latest/optiprime.html)

## 1. Installing OptiPrime

OptiPrime **cannot** be a dependency of PEGG: it needs JAX (Python 3.11+, `numpy>=2`), while PEGG pins `numpy<2` on Python 3.9/3.10 for its pickled scoring models. The two cannot share an environment, so PEGG runs OptiPrime in a **subprocess against a separate interpreter**.

Install it once, into its own conda environment. The upstream `requirements.txt` is unpinned and does *not* resolve — use these versions:

```bash
# 1. get the source
git clone https://github.com/alvin-hsu/optiprime-src ~/optiprime/src

# 2. a dedicated Python 3.11 environment
conda create -p ~/optiprime/env python=3.11 -y
conda activate ~/optiprime/env

# 3. lightgbm and numpy from conda-forge -- pip builds fail without CMake
conda install -c conda-forge lightgbm=3.3.5 numpy=1.26.4 -y

# 4. the JAX stack
pip install jax==0.4.35 jaxlib==0.4.35 flax==0.10.2 optax==0.2.4 pandas==2.2.3

# 5. rs3 and sglearn pull in conflicting pins, so install without dependencies
pip install --no-deps rs3 sglearn
pip install seqfold
```

⚠️ Do **not** install `jax>=0.10` — it requires `numpy>=2`, which conflicts with the LightGBM path `rs3` needs.

## 2. Point PEGG at your OptiPrime installation

Two environment variables. Set them **before** importing `pegg`, or pass the equivalent `optiprime_src` / `optiprime_python` parameters to `prime.run()`.

In [1]:
import os

os.environ['OPTIPRIME_SRC']    = os.path.expanduser('~/optiprime/src')
os.environ['OPTIPRIME_PYTHON'] = os.path.expanduser('~/optiprime/env/bin/python')

import pandas as pd
from pegg import prime, optiprime

# Confirm we are running the version we think we are. If you installed PEGG
# from PyPI this should say 3.1.0 or later.
import pegg
print('pegg version:', getattr(pegg, '__version__', 'unknown'))
print('pegg loaded from:', pegg.__file__)

/opt/miniconda3/envs/pegg/lib/python3.9/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


pegg version: 3.1.0
pegg loaded from: /Users/kexindong/Documents/GitHub/PEGG3.0/pegg/__init__.py


### Check that PEGG can find OptiPrime

`optiprime_available()` returns `True` only when both paths resolve. If this is `False`, fix the two paths above before going further — scoring will raise `OptiPrimeError` otherwise.

In [2]:
print('OptiPrime available:', optiprime.optiprime_available())
print()
print('Valid cell-line groups (%d):' % len(optiprime.VALID_GROUPS))
for g in optiprime.VALID_GROUPS:
    print('   ', g)
print()
print('Default group:', optiprime.DEFAULT_GROUP)

OptiPrime available: True

Valid cell-line groups (12):
    Kim_A549
    Kim_DLD1
    Kim_HCT116
    Kim_HEK293T
    Kim_HeLa
    Kim_MDA-MB-231
    Kim_NIH3T3
    Liu_HEK293T
    Liu_HeLa
    Schwank_HEK293T
    Schwank_K562
    Schwank_U2OS

Default group: Liu_HeLa


## 3. Choosing a cell line — this is an MMR decision

`optiprime_group` is **not a tuning knob**. It selects which cell line's parameters the model scores with, and that encodes **mismatch repair status**:

- **HEK293T** is partially MMR-deficient
- **HeLa** is MMR-proficient

The same pegRNA can score **0.174** under `Liu_HeLa` and **0.407** under `Liu_HEK293T`.

Per Hsu et al., pegRNAs optimised under MMR-proficient (HeLa) parameters generally transfer to HEK293T, **but the reverse is not true**. So design on `Liu_HeLa` (the default) unless you have a specific reason not to.

⚠️ **An unrecognised group name is not an error in OptiPrime.** Its rate loader leaves the group factor at zero and logs only at `INFO`, so a typo returns a plausible-looking score that means nothing. PEGG validates the name and raises instead — as shown below.

In [3]:
# PEGG rejects an invalid group name rather than silently scoring it as zero.
try:
    optiprime.score(pd.DataFrame(), group='HeLa')   # note: missing the 'Liu_' prefix
except optiprime.OptiPrimeError as e:
    print('Rejected, as it should be:')
    print(' ', str(e)[:300])

Rejected, as it should be:
  unknown OptiPrime group 'HeLa'. The released weights only carry group factors for: Kim_A549, Kim_DLD1, Kim_HCT116, Kim_HEK293T, Kim_HeLa, Kim_MDA-MB-231, Kim_NIH3T3, Liu_HEK293T, Liu_HeLa, Schwank_HEK293T, Schwank_K562, Schwank_U2OS. An unrecognised name is not rejected by OptiPrime itself -- it sil


## 4. Design and score

A single target sequence in PrimeDesign format. `optiprime=True` turns scoring on; `rankby='OptiPrime_Score'` ranks by it (and implies `optiprime=True`).

In [4]:
# PrimeDesign format: the edit is written inline as (REF/ALT).
# Include plenty of context either side -- at least ~100 nt is recommended.
seq = (
    'ATGGCGACCCTGGAAAAGCTGATGAAGGCCTTCGAGTCCCTCAAGTCCTTCCAGCAGCAGCAGCAGCAGCAG'
    'CAGCAGCAGCAGCAGCAGCAACAGCCACCTCCACCGCCGCCGCCGCCGCCGCCTCCTCCTCAGCTTCCTCAG'
    'CCGCCGCCG(C/T)AGGCACAGCCGCTGCTGCCTCAGCCGCAGCCGCCCCCGCCGCCGCCCCCGCCGCCACC'
    'CGGCCCGGCTGTGGCTGAGGAGCCGCTGCACCGACCAAAAAAAGAGCTTTCTGCTACCAAGAAAGACCGTGT'
    'GAATCATTGTCTGACAATATGTGAAAACATAGTGGCACAGTCTGTCAGAAATTCTCCA'
)

mutations = pd.DataFrame({'SEQ': [seq]})
mutations

,SEQ
0,ATGGCGACCCTGGAAAAGCTGATGAAGGCCTTCGAGTCCCTCAAGT...


In [5]:
%%time
pegRNAs = prime.run(
    mutations, 'PrimeDesign', None,
    pegRNAs_per_mut=10,
    optiprime=True,
    optiprime_group='Liu_HeLa',     # MMR-proficient; see section 3
    rankby='OptiPrime_Score',
)

print('designed %d pegRNAs' % len(pegRNAs))

OptiPrime: scoring 40/56 pegRNAs (top 40 per mutation by PEGG2_Score; set optiprime_prefilter=None to score all)


designed 10 pegRNAs
CPU times: user 389 ms, sys: 74.9 ms, total: 464 ms
Wall time: 19.5 s


### The two new columns

- **`OptiPrime_Score`** — predicted editing efficiency, roughly 0–1. Higher is better.
- **`OptiPrime_error`** — why a pegRNA was *not* scored, or `NaN` when it was.

Scoring never fails silently: an unscored pegRNA always carries a reason.

In [6]:
print(pegRNAs['OptiPrime_Score'].describe())
print()
print('unscored pegRNAs and why:')
print(pegRNAs['OptiPrime_error'].value_counts(dropna=False))

count    10.000000
mean      0.220570
std       0.015275
min       0.198380
25%       0.211956
50%       0.216479
75%       0.232744
max       0.246119
Name: OptiPrime_Score, dtype: float64

unscored pegRNAs and why:
OptiPrime_error
None    10
Name: count, dtype: int64


In [7]:
# OptiPrime and PEGG2_Score do not always agree -- that is the point of using it.
cols = ['Protospacer_30', 'RTT_length', 'PBS_length',
        'PEGG2_Score', 'OptiPrime_Score']
pegRNAs[cols].head(10)

,Protospacer_30,RTT_length,PBS_length,PEGG2_Score,OptiPrime_Score
0,CTCAGCTTCCTCAGCCGCCGCCGCAGGCAC,30,13,22.374591,0.246119
1,CTCAGCTTCCTCAGCCGCCGCCGCAGGCAC,30,8,23.197009,0.239044
2,CTCAGCTTCCTCAGCCGCCGCCGCAGGCAC,30,10,22.530170,0.237595
3,CTCAGCTTCCTCAGCCGCCGCCGCAGGCAC,25,8,22.712436,0.218193
4,CTCAGCTTCCTCAGCCGCCGCCGCAGGCAC,25,13,21.890018,0.216932
5,CTGAGGCAGCAGCGGCTGTGCCTGCGGCGG,25,13,22.951239,0.216026
6,CTCAGCTTCCTCAGCCGCCGCCGCAGGCAC,30,15,21.595128,0.213996
7,CTGAGGCAGCAGCGGCTGTGCCTGCGGCGG,25,10,22.904093,0.211277
8,CTCAGCTTCCTCAGCCGCCGCCGCAGGCAC,25,10,22.045596,0.208136
9,CTGAGGCAGCAGCGGCTGTGCCTGCGGCGG,25,8,22.253230,0.198380


## 5. Filtering by score

`optiprime_cutoff` drops pegRNAs below a threshold. It implies `optiprime=True`.

⚠️ **`run()` scores *before* the downstream polyT and sensor filters** that most pipelines apply afterwards. A mutation with 10 pegRNAs here can end up with none after those filters. If per-mutation coverage matters, check it *after* filtering, not here.

In [8]:
filtered = prime.run(
    mutations, 'PrimeDesign', None,
    pegRNAs_per_mut=10,
    optiprime=True,
    optiprime_cutoff=0.05,          # drop anything below 0.05
)

print('%d pegRNAs passed the cutoff' % len(filtered))
filtered[['RTT_length', 'PBS_length', 'OptiPrime_Score']].head()

OptiPrime: scoring 40/56 pegRNAs (top 40 per mutation by PEGG2_Score; set optiprime_prefilter=None to score all)


OptiPrime cutoff 0.05: kept 30/40 pegRNAs (0 were unscored)
10 pegRNAs passed the cutoff


,RTT_length,PBS_length,OptiPrime_Score
0,15,13,0.066530
1,15,10,0.063631
2,15,15,0.054158
3,10,13,0.057085
4,30,13,0.139097


## 6. Runtime, and how to control it

Two costs:

| | Cost | Notes |
|---|---|---|
| **Start-up** (fixed) | ~14 s | imports JAX, builds the model, JIT-compiles. Paid once per worker. |
| **Per pegRNA** (marginal) | ~0.1 s | on one CPU core |

Measured on an 8-core CPU laptop, no GPU:

| pegRNAs | Total | Per pegRNA (apparent) |
|---|---|---|
| 10 | 15 s | 1.50 s |
| 50 | 18 s | 0.37 s |
| 200 | 33 s | 0.17 s |
| 720 | 82 s | 0.11 s |

The apparent per-pegRNA cost falls as the set grows, because the fixed start-up is spread over more rows. A rough estimate:

```
total seconds  ~=  14  +  0.1 x (number of pegRNAs scored)
```

So a 10,000-pegRNA library is on the order of 15–20 minutes on one core, and substantially less on a GPU.

**Two parameters control the total:**

- **`optiprime_prefilter`** (default `4`) — score only the top `pegRNAs_per_mut x 4` candidates per mutation, ranked by `PEGG2_Score`. **Lower this first if a run is too slow.** Set to `None` to score everything.
- **`optiprime_jobs`** (default: CPU count − 1) — worker processes.

⚠️ **Sharding only engages above 500 pegRNAs per worker.** Each worker pays the full ~14 s start-up, so PEGG refuses shards smaller than that: below ~1,000 pegRNAs you get one worker regardless of `optiprime_jobs`.

Above that threshold it does help — 720 pegRNAs took 82 s on one worker and 53 s on two (1.5x). The speedup is sublinear because each extra worker repeats the start-up, so it is worth setting only for large libraries.

In [9]:
# Scoring every candidate rather than the top 4x. Slower, and it rarely changes
# which pegRNAs come out on top -- shown here so the cost is visible.
#
# Note the printed "scoring N/M" line rather than the length of the result:
# run() returns only the pegRNAs_per_mut it keeps, so the returned frame does
# not show how many were actually scored.
%time all_scored = prime.run(mutations, 'PrimeDesign', None, \
                             pegRNAs_per_mut=10, optiprime=True, \
                             optiprime_prefilter=None)
print('returned %d pegRNAs (all scored candidates are counted in the line above)'
      % len(all_scored))

CPU times: user 78.9 ms, sys: 45.1 ms, total: 124 ms
Wall time: 23 s
returned 10 pegRNAs (all scored candidates are counted in the line above)


## 7. Silent bystanders + OptiPrime

When `silent_bystander=True` and `optiprime=True` are combined, PEGG scores the sequence each pegRNA **actually installs** — the intended edit plus its bystanders — not just the intended edit.

`cap_total_per_mut` (default `True`) splits the per-mutation budget between plain and bystander designs rather than taking the overall top N.

**Why that matters:** plain and bystander designs do not share a score scale. A bystander can disrupt the PAM and shift RTT GC content, both weighted positively by the model, so capping by overall rank lets bystander designs sweep the top — and a variant whose only viable designs are plain loses every row.

In [10]:
# PrimeDesign input carries no genomic coordinates, so the reading frame must be
# declared with ORF_start (0, 1 or 2) and the sequence must be entirely coding.
# For cBioPortal input, use bystander.cds_for_variants() instead.
with_bystanders = prime.run(
    mutations, 'PrimeDesign', None,
    pegRNAs_per_mut=6,
    silent_bystander=True,
    silent_per_mut=2,
    ORF_start=0,            # this context begins at the ATG
    seed=0,                 # reproducible bystander choice
    optiprime=True,
)

print('%d pegRNAs (%d carry bystanders)'
      % (len(with_bystanders), int(with_bystanders['has_silent_bystander'].sum())))

with_bystanders[['has_silent_bystander', 'n_bystander_muts',
                 'PEGG2_Score', 'OptiPrime_Score']].head(12)

Note: input_format='PrimeDesign' carries no genomic coordinates, so CDS membership and splice sites cannot be checked. The input sequence is assumed to be entirely coding and in frame from offset ORF_start=0.
OptiPrime: scoring 48/136 pegRNAs (top 24 per mutation and design type by PEGG2_Score; set optiprime_prefilter=None to score all)


6 pegRNAs (3 carry bystanders)


,has_silent_bystander,n_bystander_muts,PEGG2_Score,OptiPrime_Score
0,True,1,24.571959,0.245202
1,True,1,24.045330,0.205752
2,True,1,23.917284,0.071495
3,False,0,23.439271,0.019607
4,False,0,23.411964,0.066530
5,False,0,23.392125,0.018761


In [11]:
# Plain and bystander designs score on different scales -- compare them directly.
with_bystanders.groupby('has_silent_bystander')['OptiPrime_Score'].describe()

,count,mean,std,min,25%,50%,75%,max
has_silent_bystander,,,,,,,,
False,3.0,0.034966,0.027339,0.018761,0.019184,0.019607,0.043068,0.066530
True,3.0,0.174149,0.091063,0.071495,0.138623,0.205752,0.225477,0.245202


## Citation

If you use OptiPrime scoring, please cite **both** PEGG and OptiPrime:

> Gould, S.I., Wuest, A.N., Dong, K. et al. High-throughput evaluation of genetic variants with prime editing sensor libraries. *Nat Biotechnol* (2024). https://doi.org/10.1038/s41587-024-02172-9

> Hsu, A. et al. Massively parallel measurement of prime editing outcomes enables predictive modelling of pegRNA efficiency. *Nat Biotechnol* (2026). https://doi.org/10.1038/s41587-026-03261-7